# Preparar el dataset (Roboflow + fotos propias) → `dataset_tornilleria.zip`

Este notebook hace **todo el trabajo de preparación en Colab**, sin instalar nada en tu computador:

1. Recibe los `.zip` que descargaste de Roboflow (formato YOLOv8).
2. Recorta cada pieza y la guarda en la carpeta de su clase.
3. Recibe tu `propio.zip` con tus fotos.
4. Entrega `dataset_tornilleria.zip`, que es el que subes al notebook de entrenamiento.

**Ejecuta cada celda con el botón ▶, en orden, y espera a que termine antes de seguir.**

## Paso 1. Configuración (aquí solo tocas el mapa si los nombres no coinciden)

In [1]:
# Mapa  "nombre en Roboflow = nombre de tu carpeta".  No importan mayúsculas/minúsculas.
# Si en el Paso 4 ves que un nombre no coincide con el data.yaml, corrígelo aquí.
MAPA = "Screw=tornillo,Nut=tuerca,Washer=arandela,LongScrew=tornillo_largo,Hexagonal Bolt=perno_hex"

MAX_POR_CLASE = 300   # tope de recortes públicos por clase (evita desbalance)
CLASES = ["tuerca", "arandela", "tornillo", "tornillo_largo", "perno_hex"]
print("Configuración lista.")

Configuración lista.


## Paso 2. Instalar el script de recorte (se crea solo, no subas nada)

In [2]:
SCRIPT = r'''#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
recortar_roboflow.py
====================
Convierte datasets de DETECCIÓN de Roboflow (exportados en formato YOLO) en un
dataset de CLASIFICACIÓN: recorta cada caja anotada y la guarda en una carpeta
con el nombre de su clase.

Cada recorte se pasa a cuadrado con relleno NEGRO (para parecerse a tu fomi
negro mate) y se redimensiona a 224x224.

Uso rápido
----------
1) En Roboflow Universe abre el dataset -> Download Dataset -> formato
   "YOLOv8" (o YOLOv5/v9) -> descarga el .zip y descomprímelo.
   Debe quedar una carpeta con data.yaml y subcarpetas train/ valid/ test/.

2) Ejecuta (puedes repetir --src varias veces con distintos datasets):

   python recortar_roboflow.py \
       --src ./screw-nut-washer \
       --src ./irp_dataset_2 \
       --dst ./dataset_tornilleria/publico \
       --map "Nut=tuerca,Washer=arandela,LongScrew=tornillo_largo,Hexagonal Bolt=perno_hex"

   Solo se guardan las clases que aparezcan en --map; las demás se ignoran.
   Los nombres a la izquierda son los de data.yaml (no distingue mayúsculas).
   Varias clases de origen pueden ir a la misma clase destino, por ejemplo:
   "silver nut=tuerca,black nut=tuerca,silver washer=arandela,black washer=arandela"

3) En Colab: sube el script y ejecútalo con "!python recortar_roboflow.py ..."

Requisitos: pip install pillow pyyaml
"""

import argparse
import sys
from pathlib import Path

try:
    import yaml
    from PIL import Image
except ImportError:
    sys.exit("Faltan librerías. Ejecuta: pip install pillow pyyaml")

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SPLITS = ["train", "valid", "val", "test"]


def parse_map(texto):
    """'Nut=tuerca,Washer=arandela' -> {'nut': 'tuerca', 'washer': 'arandela'}"""
    mapa = {}
    for par in texto.split(","):
        if "=" not in par:
            sys.exit(f"Formato inválido en --map: '{par}'. Usa Origen=destino")
        origen, destino = par.split("=", 1)
        mapa[origen.strip().lower()] = destino.strip()
    return mapa


def cargar_nombres(src: Path):
    """Lee la lista de clases de data.yaml (acepta lista o diccionario)."""
    yml = src / "data.yaml"
    if not yml.exists():
        candidatos = list(src.glob("*.yaml")) + list(src.glob("*.yml"))
        if not candidatos:
            sys.exit(f"No encontré data.yaml en {src}")
        yml = candidatos[0]
    with open(yml, "r", encoding="utf-8") as f:
        datos = yaml.safe_load(f)
    nombres = datos["names"]
    if isinstance(nombres, dict):
        nombres = [nombres[k] for k in sorted(nombres, key=lambda x: int(x))]
    return list(nombres)


def leer_etiqueta(linea):
    """
    Una línea YOLO puede ser:
      caja:     clase cx cy w h
      polígono: clase x1 y1 x2 y2 ... (segmentación)
    Devuelve (clase, (x1, y1, x2, y2)) normalizado 0-1, o None si es inválida.
    """
    partes = linea.strip().split()
    if len(partes) < 5:
        return None
    clase = int(float(partes[0]))
    v = [float(x) for x in partes[1:]]
    if len(v) == 4:
        cx, cy, w, h = v
        return clase, (cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2)
    xs, ys = v[0::2], v[1::2]
    return clase, (min(xs), min(ys), max(xs), max(ys))


def recorte_cuadrado(img, caja_norm, margen, lado_final):
    """Recorta la caja (con margen), la centra en un lienzo negro cuadrado y redimensiona."""
    W, H = img.size
    x1, y1, x2, y2 = caja_norm
    x1, x2 = x1 * W, x2 * W
    y1, y2 = y1 * H, y2 * H
    bw, bh = x2 - x1, y2 - y1
    x1 = max(0, x1 - bw * margen)
    x2 = min(W, x2 + bw * margen)
    y1 = max(0, y1 - bh * margen)
    y2 = min(H, y2 + bh * margen)
    if x2 - x1 < 2 or y2 - y1 < 2:
        return None
    recorte = img.crop((int(x1), int(y1), int(x2), int(y2)))
    lado = max(recorte.size)
    lienzo = Image.new("RGB", (lado, lado), (0, 0, 0))
    lienzo.paste(recorte, ((lado - recorte.width) // 2, (lado - recorte.height) // 2))
    return lienzo.resize((lado_final, lado_final), Image.LANCZOS)


def carpetas_split(src: Path):
    """Devuelve pares (carpeta_imagenes, carpeta_etiquetas) de cada split."""
    pares = []
    for s in SPLITS:
        base = src / s
        if not base.exists():
            continue
        if (base / "images").exists() and (base / "labels").exists():
            pares.append((s, base / "images", base / "labels"))
        else:  # estructura plana
            pares.append((s, base, base))
    if not pares:  # sin splits: todo en la raíz
        if (src / "images").exists() and (src / "labels").exists():
            pares.append(("all", src / "images", src / "labels"))
        else:
            pares.append(("all", src, src))
    return pares


def buscar_imagen(carpeta_img: Path, stem: str):
    for ext in IMG_EXT:
        for e in (ext, ext.upper()):
            p = carpeta_img / f"{stem}{e}"
            if p.exists():
                return p
    return None


def procesar(src: Path, dst: Path, mapa, margen, tam, min_px, max_por_clase, contadores):
    nombres = cargar_nombres(src)
    print(f"\n[{src.name}] clases en data.yaml: {nombres}")
    ignoradas = [n for n in nombres if n.lower() not in mapa]
    if ignoradas:
        print(f"   (se ignoran por no estar en --map: {ignoradas})")
    prefijo = src.name.replace(" ", "_")

    for split, carpeta_img, carpeta_lbl in carpetas_split(src):
        for lbl in sorted(carpeta_lbl.glob("*.txt")):
            ruta_img = buscar_imagen(carpeta_img, lbl.stem)
            if ruta_img is None:
                continue
            with open(lbl, "r", encoding="utf-8") as f:
                lineas = [l for l in f.read().splitlines() if l.strip()]
            if not lineas:
                continue
            img = None
            for k, linea in enumerate(lineas):
                dato = leer_etiqueta(linea)
                if dato is None:
                    continue
                idx, caja = dato
                if idx >= len(nombres):
                    continue
                destino = mapa.get(nombres[idx].lower())
                if destino is None:
                    continue
                if max_por_clase and contadores.get(destino, 0) >= max_por_clase:
                    continue
                if img is None:
                    img = Image.open(ruta_img).convert("RGB")
                W, H = img.size
                if min((caja[2] - caja[0]) * W, (caja[3] - caja[1]) * H) < min_px:
                    continue  # caja demasiado pequeña, recorte inútil
                salida = recorte_cuadrado(img, caja, margen, tam)
                if salida is None:
                    continue
                carpeta = dst / destino
                carpeta.mkdir(parents=True, exist_ok=True)
                nombre = f"{prefijo}_{split}_{lbl.stem}_{k}.jpg"
                salida.save(carpeta / nombre, quality=95)
                contadores[destino] = contadores.get(destino, 0) + 1


def main():
    ap = argparse.ArgumentParser(description="Recorta datasets YOLO de Roboflow por clase.")
    ap.add_argument("--src", action="append", required=True,
                    help="Carpeta del dataset descomprimido (repetible).")
    ap.add_argument("--dst", required=True, help="Carpeta de salida (una subcarpeta por clase).")
    ap.add_argument("--map", required=True,
                    help='Mapeo origen=destino separado por comas. Ej: "Nut=tuerca,Washer=arandela"')
    ap.add_argument("--margen", type=float, default=0.10, help="Margen extra alrededor de la caja (0.10 = 10%%).")
    ap.add_argument("--tam", type=int, default=224, help="Lado final de la imagen cuadrada.")
    ap.add_argument("--min-px", type=int, default=24, help="Lado mínimo (px) de la caja original.")
    ap.add_argument("--max-por-clase", type=int, default=0,
                    help="Máximo de recortes por clase destino (0 = sin límite).")
    args = ap.parse_args()

    mapa = parse_map(args.map)
    dst = Path(args.dst)
    contadores = {}
    for s in args.src:
        src = Path(s)
        if not src.exists():
            sys.exit(f"No existe la carpeta: {src}")
        procesar(src, dst, mapa, args.margen, args.tam, args.min_px, args.max_por_clase, contadores)

    print("\n=== Resumen ===")
    if not contadores:
        print("No se generó ningún recorte. Revisa los nombres en --map contra data.yaml.")
    for clase, n in sorted(contadores.items()):
        print(f"  {clase:<18} {n:>5} recortes -> {dst / clase}")


if __name__ == "__main__":
    main()
'''
open('recortar_roboflow.py', 'w', encoding='utf-8').write(SCRIPT)
print('recortar_roboflow.py creado.')

recortar_roboflow.py creado.


## Paso 3. Subir los .zip de Roboflow
Al ejecutar la celda aparece un botón **"Seleccionar archivos"**. Elige **todos** los `.zip` de Roboflow a la vez
(por ejemplo el de *screw-nut-washer* y el de *screwdetectclassification*).

In [4]:
import os, zipfile, shutil, glob
from google.colab import files

subidos = files.upload()
shutil.rmtree("/content/roboflow", ignore_errors=True)
os.makedirs("/content/roboflow")
for nombre in subidos:
    destino = "/content/roboflow/" + os.path.splitext(nombre)[0].replace(" ", "_")
    with zipfile.ZipFile(nombre) as z:
        z.extractall(destino)
    print("Descomprimido:", nombre, "->", destino)

Saving screw-nut-washer.zip to screw-nut-washer.zip
Saving screwdetectclassification-xrrbi-gdsj.zip to screwdetectclassification-xrrbi-gdsj.zip
Descomprimido: screw-nut-washer.zip -> /content/roboflow/screw-nut-washer
Descomprimido: screwdetectclassification-xrrbi-gdsj.zip -> /content/roboflow/screwdetectclassification-xrrbi-gdsj


## Paso 4. Revisar los nombres de clase de cada dataset
Compara lo que sale aquí con el `MAPA` del Paso 1. Los nombres a la izquierda del `=` deben aparecer en esta lista.

In [5]:
import yaml

def encontrar_raiz(carpeta):
    for raiz, dirs, archivos in os.walk(carpeta):
        if "data.yaml" in archivos:
            return raiz
    return None

RAICES = []
for carpeta in sorted(glob.glob("/content/roboflow/*")):
    raiz = encontrar_raiz(carpeta)
    if raiz is None:
        print(f"⚠️ {os.path.basename(carpeta)}: no encontré data.yaml (¿descargaste el formato YOLOv8?)")
        continue
    nombres = yaml.safe_load(open(os.path.join(raiz, "data.yaml"), encoding="utf-8"))["names"]
    if isinstance(nombres, dict):
        nombres = list(nombres.values())
    RAICES.append(raiz)
    print(f"{os.path.basename(carpeta)}: {list(nombres)}")

screw-nut-washer: ['LongScrew', 'Nut', 'Screw', 'Washer']
screwdetectclassification-xrrbi-gdsj: ['Hex Washer Screw', 'Hexagonal Bolt', 'Philips Screw', 'Pozidriv Screw', 'Torx Screw']


## Paso 5. Recortar las piezas por clase

In [6]:
import subprocess, sys

PUBLICO = "/content/dataset_tornilleria/publico"
shutil.rmtree("/content/dataset_tornilleria", ignore_errors=True)

cmd = [sys.executable, "recortar_roboflow.py", "--dst", PUBLICO, "--map", MAPA,
       "--max-por-clase", str(MAX_POR_CLASE)]
for r in RAICES:
    cmd += ["--src", r]
resultado = subprocess.run(cmd, capture_output=True, text=True)
print(resultado.stdout)
print(resultado.stderr)


[screw-nut-washer] clases en data.yaml: ['LongScrew', 'Nut', 'Screw', 'Washer']

[screwdetectclassification-xrrbi-gdsj] clases en data.yaml: ['Hex Washer Screw', 'Hexagonal Bolt', 'Philips Screw', 'Pozidriv Screw', 'Torx Screw']
   (se ignoran por no estar en --map: ['Hex Washer Screw', 'Philips Screw', 'Pozidriv Screw', 'Torx Screw'])

=== Resumen ===
  arandela             300 recortes -> /content/dataset_tornilleria/publico/arandela
  perno_hex            253 recortes -> /content/dataset_tornilleria/publico/perno_hex
  tornillo             300 recortes -> /content/dataset_tornilleria/publico/tornillo
  tornillo_largo       262 recortes -> /content/dataset_tornilleria/publico/tornillo_largo
  tuerca               300 recortes -> /content/dataset_tornilleria/publico/tuerca




In [7]:
# Resumen: cuántos recortes hay por clase (cada clase debería tener al menos ~50)
for c in CLASES:
    carpeta = f"{PUBLICO}/{c}"
    n = len(os.listdir(carpeta)) if os.path.isdir(carpeta) else 0
    aviso = "" if n >= 50 else "   ⚠️ pocas imágenes: revisa el MAPA o agrega otro dataset"
    print(f"{c:<16}{n:>5}{aviso}")

tuerca            300
arandela          300
tornillo          300
tornillo_largo    262
perno_hex         253


## Paso 6. Subir TUS fotos (`propio.zip`)
Crea en tu computador una carpeta llamada `propio` con una subcarpeta por clase
(`tuerca`, `arandela`, `tornillo`, `tornillo_largo`, `perno_hex`), comprímela como **`propio.zip`** y súbela aquí.
Al abrir el zip debe verse directamente la carpeta `propio`.

In [8]:
subidos = files.upload()
with zipfile.ZipFile(list(subidos.keys())[0]) as z:
    z.extractall("/content/_propio_tmp")

# Localiza la carpeta 'propio' (por si el zip trae una carpeta extra encima)
origen = None
for raiz, dirs, _ in os.walk("/content/_propio_tmp"):
    if "propio" in dirs:
        origen = os.path.join(raiz, "propio"); break
if origen is None:
    origen = "/content/_propio_tmp"   # el zip traía directamente las carpetas de clase
shutil.copytree(origen, "/content/dataset_tornilleria/propio")

print(f"{'clase':<16}{'propias':>8}")
for c in CLASES:
    carpeta = f"/content/dataset_tornilleria/propio/{c}"
    n = len(os.listdir(carpeta)) if os.path.isdir(carpeta) else 0
    aviso = "" if n >= 20 else "   ⚠️ faltan fotos (mínimo 20, ideal 80)"
    print(f"{c:<16}{n:>8}{aviso}")

Saving propio.zip to propio.zip
clase            propias
tuerca                31
arandela              31
tornillo              31
tornillo_largo        31
perno_hex             31


## Paso 7. Crear y descargar `dataset_tornilleria.zip`

In [9]:
shutil.make_archive("/content/dataset_tornilleria", "zip", "/content", "dataset_tornilleria")
print("Tamaño del zip: %.1f MB" % (os.path.getsize("/content/dataset_tornilleria.zip") / 1e6))
files.download("/content/dataset_tornilleria.zip")

Tamaño del zip: 20.8 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ **Listo.** Ahora abre `Entrenamiento_Tornilleria_RPi4.ipynb` y, cuando te pida el zip, sube este `dataset_tornilleria.zip`.